In [1]:
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.exceptions import InvalidSignature

import hashlib

In [2]:
#GENERATING ECDSA KEY PAIRS

# Generate Alice's private key using the SECP256K1 elliptic curve
alice_private_key = ec.generate_private_key(
    ec.SECP256K1()
)

# Derive Alice's public key from her private key
alice_public_key = alice_private_key.public_key()


# Generate Bob's private key using the same elliptic curve
bob_private_key = ec.generate_private_key(
    ec.SECP256K1()
)

# Derive Bob's public key from his private key
bob_public_key = bob_private_key.public_key()

print("ECDSA key pairs generated successfully.")
print("Alice and Bob now each have a private key and a public key.")

ECDSA key pairs generated successfully.
Alice and Bob now each have a private key and a public key.


In [3]:
#SERIALIZING THE PUBLIC KEYS

# Convert Alice's public key into bytes
alice_public_bytes = alice_public_key.public_bytes(
    encoding=serialization.Encoding.X962,
    format=serialization.PublicFormat.UncompressedPoint
)

# Convert Bob's public key into bytes
bob_public_bytes = bob_public_key.public_bytes(
    encoding=serialization.Encoding.X962,
    format=serialization.PublicFormat.UncompressedPoint
)

# Display the public keys in hexadecimal format
print("\nAlice's Public Key:")
print(alice_public_bytes.hex())

print("\nBob's Public Key:")
print(bob_public_bytes.hex())


Alice's Public Key:
04d75940227f1498a5355d7895948e92b229ceff3cff59331deb6a3fd9210ff017b75647b697f03f8abe6f2f4b88d3a2cadbb88f1b60cd026ae93ccd782dd6bcb9

Bob's Public Key:
047675263f7b82be227d24e74154e2e064ae5356bba9d28757f2519fa3b72739709f0fb198b67d05be60b9e1db09268d6b16650d8eee01f1614c3a9c099e8ffb4d


In [4]:
#DERIVING THE SIMPLIFIED WALLET ADDRESSES

def derive_address(public_key_bytes):
    """
    Derive a simplified wallet address from a public key.
    
    Steps:
    1. Hash the public key using SHA-256.
    2. Hash the SHA-256 result using RIPEMD-160.
    3. Add a simple prefix to identify it as a demo address.
    """

    # Step 1: Apply SHA-256 to the public key
    sha256_hash = hashlib.sha256(public_key_bytes).digest()

    # Step 2: Apply RIPEMD-160 to the SHA-256 hash
    ripemd160_hash = hashlib.new(
        "ripemd160",
        sha256_hash
    ).digest()

    # Step 3: Convert the final hash to hexadecimal
    address = "WALLET_" + ripemd160_hash.hex()

    return address


# Derive Alice's simplified wallet address
alice_address = derive_address(alice_public_bytes)

# Derive Bob's simplified wallet address
bob_address = derive_address(bob_public_bytes)


# Display the wallet addresses
print("\nAlice's Simplified Wallet Address:")
print(alice_address)

print("\nBob's Simplified Wallet Address:")
print(bob_address)


Alice's Simplified Wallet Address:
WALLET_e4230a511f088ac47d0329f4206143d772b94c14

Bob's Simplified Wallet Address:
WALLET_4dc648c32360eb70a6becff5d533d1146ac89140


In [5]:
#CREATE AND SIGN A TRANSACTION

# Create a transaction payload
transaction = {
    "sender": alice_address,
    "recipient": bob_address,
    "amount": 10,
    "currency": "DemoCoin"
}

# Convert the transaction into a consistent byte format
transaction_payload = str(transaction).encode("utf-8")

print("\nOriginal Transaction:")
print(transaction)

# Alice signs the transaction using her private key
signature = alice_private_key.sign(
    transaction_payload,
    ec.ECDSA(hashes.SHA256())
)

# Display the digital signature
print("\nDigital Signature:")
print(signature.hex())


Original Transaction:
{'sender': 'WALLET_e4230a511f088ac47d0329f4206143d772b94c14', 'recipient': 'WALLET_4dc648c32360eb70a6becff5d533d1146ac89140', 'amount': 10, 'currency': 'DemoCoin'}

Digital Signature:
304502203e74779065e4fb37d338f03751e6209ec037257cc43df33e2eea8d81f7bd350d022100c73ff93a7419c1434266be05266edf5e71c5451f87bbe872877b6bbf18ed35aa


In [6]:
# VERIFICATION OF THE ORIGINAL TRANSACTION

try:
    alice_public_key.verify(
        signature,
        transaction_payload,
        ec.ECDSA(hashes.SHA256())
    )

    print("\nSignature verification successful!")
    print("The original transaction has not been altered.")

except InvalidSignature:

    print("\nSignature verification failed!")


Signature verification successful!
The original transaction has not been altered.
